In [8]:
path="/data_lake/gold/intlpris/"

In [9]:
from pyspark.sql import SparkSession
from impala.dbapi import connect
import pandas as pd
import os
import json
import requests
from pyspark.sql import functions as sf, types as T
import sys
from datetime import datetime, timedelta

spark = SparkSession.builder \
            .appName("Inteligência Prisional") \
            .config("spark.dynamicAllocation.enabled", "true") \
            .config("spark.dynamicAllocation.initialExecutors", "1") \
            .config("spark.dynamicAllocation.minExecutors", "1") \
            .config("spark.dynamicAllocation.maxExecutors", "4") \
            .config("spark.executor.memory", "4g") \
            .config("spark.executor.cores", "1") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.cores", "1") \
            .config("spark.yarn.executor.memoryOverhead", "1g") \
            .config("spark.executor.memoryOverhead", "1g") \
            .config("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY") \
            .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
            .master('local') \
            .enableHiveSupport() \
            .getOrCreate()

In [3]:
def get_dtype_df(dataframe):
    string_cols = [col for col, dtype in dataframe.dtypes if dtype == "string"]
    
    # max em BYTES (UTF-8) para cada coluna string
    if string_cols:
        df_max_bytes = (
            dataframe.select([
                sf.max(sf.length(sf.encode(sf.col(col), "UTF-8"))).alias(col)
                for col in string_cols
            ])
            .first()
            .asDict()
        )
    else:
        df_max_bytes = {}
    
    df_schema = pd.json_normalize(json.loads(dataframe.schema.json()), record_path=['fields'])[['name', 'type']]

    def adjust_dtype(name, dtype):
        if dtype == "long":
            return "bigint"
        
        if dtype == 'string':
            max_length = df_max_bytes.get(name) or 1
            
            # ajusta para CHAR(n) se for até 16 caracteres, senão usa VARCHAR(n)
            if max_length <= 16:
                return f"char({max_length})"  # garantir CHAR(1) como mínimo
            else:
                return f"varchar({max_length})" 
            
        else:
            return dtype

    df_schema['type'] = df_schema.apply(lambda x: adjust_dtype(x['name'], x['type']), axis=1)
    return df_schema

def write_impala_table_partioned(df, impala_schema, impala_table, br_path):
    dtype_df = get_dtype_df(df)
    dtypes_sting = ",\n    ".join([f"{row['name']} {row['type']}" for index, row in dtype_df.iterrows()])
    creation_query = f"""CREATE EXTERNAL TABLE {impala_schema}.{impala_table} 
                        ({dtypes_sting})
                        STORED AS PARQUET LOCATION '{br_path}'
                        """
    drop_query = f"""DROP TABLE {impala_schema}.{impala_table} """
    conn = connect(host='worker03-prod.sejus.es.gov.br', 
                   port=21050, 
                   database='bronze', 
                   auth_mechanism='GSSAPI',
                   kerberos_service_name='impala',
                   use_ssl=True,
                   ca_cert="/var/lib/cloudera-scm-agent/agent-cert/cm-auto-global_cacerts.pem")

    cursor = conn.cursor()
    print(creation_query)
    print(drop_query)
    try:
        cursor.execute(drop_query)
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
    except: 
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
        
def enviar_gold_para_postgres(nome_tabela_origem, pk_postgres):
    """
    Cria/substitui uma tabela no PostgreSQL a partir de uma tabela Spark/Hive
    e define uma chave primária no campo informado, caso pk_postgres seja informado.

    Parâmetros:
        nome_tabela_origem : str
            Nome completo da tabela no Spark, ex: "gold.sinp_pres_loc_atual"

        pk_postgres : str
            Nome do campo que será chave primária no PostgreSQL.
            Se vier "" ou None, a tabela será criada sem PK.

    Premissas:
        - a tabela de origem já existe no Spark
        - o driver JDBC do PostgreSQL está disponível no cluster
        - o schema/tabela de destino no PostgreSQL terão o mesmo nome da origem
          Ex: gold.sinp_pres_loc_atual -> schema "sinp", tabela "sinp_pres_loc_atual"
    """

    url = "jdbc:postgresql://10.242.38.126:5432/sinp_db"
    usuario = "usr_sinp"
    senha = "u9oLzKOato#nksFZ"
    driver = "org.postgresql.Driver"

    partes = nome_tabela_origem.split(".")
    if len(partes) != 2:
        raise ValueError("Informe a tabela no formato schema.tabela. Ex: gold.sinp_pres_loc_atual")

    schema_destino = "sinp"
    tabela_destino = partes[1]

    spark.catalog.clearCache()
    spark.sql(f"REFRESH TABLE {nome_tabela_origem}")
    df = spark.table(nome_tabela_origem).cache()
    df.count()

    colunas_df = df.columns

    tem_pk = pk_postgres is not None and str(pk_postgres).strip() != ""
    pk_postgres = str(pk_postgres).strip() if pk_postgres is not None else ""

    if tem_pk and pk_postgres not in colunas_df:
        raise ValueError(f"A PK '{pk_postgres}' não existe na tabela de origem. Colunas disponíveis: {colunas_df}")

    def mapear_tipo_postgres(campo):
        tipo = campo.dataType.simpleString().lower()

        if tipo.startswith("string"):
            return "varchar"
        elif tipo.startswith("int"):
            return "integer"
        elif tipo.startswith("bigint") or tipo.startswith("long"):
            return "bigint"
        elif tipo.startswith("double"):
            return "double precision"
        elif tipo.startswith("float"):
            return "real"
        elif tipo.startswith("boolean"):
            return "boolean"
        elif tipo.startswith("timestamp"):
            return "timestamp"
        elif tipo.startswith("date"):
            return "date"
        elif tipo.startswith("smallint"):
            return "smallint"
        elif tipo.startswith("decimal"):
            return tipo.replace("decimal", "numeric")
        else:
            return "text"

    ddl_colunas = []
    for campo in df.schema.fields:
        nome_coluna = campo.name
        tipo_pg = mapear_tipo_postgres(campo)
        ddl_colunas.append(f'"{nome_coluna}" {tipo_pg}')

    ddl_create_schema = f'create schema if not exists "{schema_destino}"'
    ddl_drop_table = f'drop table if exists "{schema_destino}"."{tabela_destino}"'

    if tem_pk:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)},
                constraint pk_{tabela_destino} primary key ("{pk_postgres}")
            )
        '''
    else:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)}
            )
        '''

    jvm = spark._sc._gateway.jvm
    jvm.java.lang.Class.forName(driver)

    conn = jvm.java.sql.DriverManager.getConnection(url, usuario, senha)
    stmt = conn.createStatement()

    try:
        stmt.execute(ddl_create_schema)
        stmt.execute(ddl_drop_table)
        stmt.execute(ddl_create_table)
    finally:
        stmt.close()
        conn.close()

    propriedades = {
        "user": usuario,
        "password": senha,
        "driver": driver
    }

    df.write \
        .mode("append") \
        .jdbc(
            url=url,
            table=f'{schema_destino}.{tabela_destino}',
            properties=propriedades
        )

    print(f"Tabela enviada com sucesso para o PostgreSQL: {schema_destino}.{tabela_destino}")
    if tem_pk:
        print(f"PK definida: {pk_postgres}")
    else:
        print("Tabela criada sem PK.")

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.storagelevel import StorageLevel

import os
import glob
import csv

# =============================================================================
# INGESTÃO BRONZE - CEP ABERTO ES
#
# Premissa:
# - Este script .py está no mesmo diretório dos CSVs extraídos.
#
# Entrada:
# - arquivos .csv do CEP Aberto ES
#
# Saída:
# - bronze.tbl_cep_es
#
# Não usa ZIP.
# Não usa API.
# Não usa Postgres.
# Não usa write_impala_table_partioned.
# =============================================================================

# -----------------------------------------------------------------------------
# Configurações
# -----------------------------------------------------------------------------

schema_destino = "bronze"
tabela_destino = "tbl_cep_es"
tabela_full = f"{schema_destino}.{tabela_destino}"

try:
    diretorio_script = os.path.dirname(os.path.abspath(__file__))
except NameError:
    diretorio_script = os.getcwd()

pasta_csv = diretorio_script

path_bronze = "/data_lake/bronze/intlpris/"
path_destino = f"{path_bronze.rstrip('/')}/{tabela_destino}"

print("=" * 120)
print("CONFIGURAÇÃO DA CARGA")
print("=" * 120)
print(f"Diretório CSV : {pasta_csv}")
print(f"Tabela destino: {tabela_full}")
print(f"Path destino  : {path_destino}")

# -----------------------------------------------------------------------------
# Localizar CSVs no mesmo diretório do script
# -----------------------------------------------------------------------------

arquivos_csv = sorted(glob.glob(os.path.join(pasta_csv, "*.csv")))

if not arquivos_csv:
    raise Exception(f"Nenhum arquivo CSV encontrado em: {pasta_csv}")

print("\n" + "=" * 120)
print("ARQUIVOS CSV LOCALIZADOS")
print("=" * 120)

for arq in arquivos_csv:
    print(arq)

# -----------------------------------------------------------------------------
# Leitura local controlada dos CSVs
# Estrutura esperada:
# 0 cep
# 1 logradouro
# 2 complemento
# 3 bairro
# 4 cidade_cepaberto_id
# 5 estado_cepaberto_id
# -----------------------------------------------------------------------------

linhas = []

for arquivo_csv in arquivos_csv:
    nome_csv = os.path.basename(arquivo_csv)

    with open(arquivo_csv, "r", encoding="utf-8", newline="") as f:
        leitor = csv.reader(f)

        for row in leitor:
            if not row:
                continue

            row = row + [None] * (6 - len(row))

            linhas.append((
                row[0],
                row[1],
                row[2],
                row[3],
                row[4],
                row[5],
                nome_csv
            ))

schema_cep = StructType([
    StructField("cep_raw", StringType(), True),
    StructField("logradouro_raw", StringType(), True),
    StructField("complemento_raw", StringType(), True),
    StructField("bairro_raw", StringType(), True),
    StructField("cidade_cepaberto_id", StringType(), True),
    StructField("estado_cepaberto_id", StringType(), True),
    StructField("arquivo_csv_origem", StringType(), True)
])

df_cepaberto_es_raw = spark.createDataFrame(linhas, schema=schema_cep)

# -----------------------------------------------------------------------------
# Função de normalização textual técnica
# -----------------------------------------------------------------------------

def normaliza_expr(expr):
    c = F.upper(F.trim(expr.cast("string")))

    c = F.translate(
        c,
        "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑáàâãäéèêëíìîïóòôõöúùûüçñ",
        "AAAAAEEEEIIIIOOOOOUUUUCNAAAAAEEEEIIIIOOOOOUUUUCN"
    )

    c = F.regexp_replace(c, r"[\r\n\t]+", " ")
    c = F.regexp_replace(c, r"\s+", " ")
    c = F.trim(c)

    c = F.when(
        c.isNull()
        | (c == "")
        | (c.isin("NULL", "[NULL]", "NULO", "NA", "N/A", "-", ".", "0")),
        F.lit(None)
    ).otherwise(c)

    return c

# -----------------------------------------------------------------------------
# Normalização mínima da referência
# -----------------------------------------------------------------------------

df_cepaberto_es = (
    df_cepaberto_es_raw
    .withColumn("cep", F.regexp_replace(F.col("cep_raw"), r"[^0-9]", ""))
    .withColumn("logradouro_norm", normaliza_expr(F.col("logradouro_raw")))
    .withColumn("complemento_norm", normaliza_expr(F.col("complemento_raw")))
    .withColumn("bairro_norm", normaliza_expr(F.col("bairro_raw")))
    .withColumn("cidade_cepaberto_id", F.trim(F.col("cidade_cepaberto_id")))
    .withColumn("estado_cepaberto_id", F.trim(F.col("estado_cepaberto_id")))
    .filter(F.length("cep") == 8)
)

# -----------------------------------------------------------------------------
# Normalização de tipo e nome do logradouro
# -----------------------------------------------------------------------------

df_cepaberto_es = (
    df_cepaberto_es
    .withColumn("logradouro_work", F.col("logradouro_norm"))

    .withColumn("logradouro_work", F.regexp_replace(F.col("logradouro_work"), r"^RUA\s*:\s*", "RUA "))
    .withColumn("logradouro_work", F.regexp_replace(F.col("logradouro_work"), r"^R\s*:\s*", "RUA "))
    .withColumn("logradouro_work", F.regexp_replace(F.col("logradouro_work"), r"^AV\s*\.?\s*:\s*", "AVENIDA "))
    .withColumn("logradouro_work", F.regexp_replace(F.col("logradouro_work"), r"^AVENIDA\s*:\s*", "AVENIDA "))
    .withColumn("logradouro_work", F.regexp_replace(F.col("logradouro_work"), r"\s+", " "))
    .withColumn("logradouro_work", F.trim(F.col("logradouro_work")))

    .withColumn(
        "tp_logradouro_norm",
        F.when(F.col("logradouro_work").rlike(r"^RUA\b|^R\b"), F.lit("RUA"))
         .when(F.col("logradouro_work").rlike(r"^AVENIDA\b|^AV\b"), F.lit("AVENIDA"))
         .when(F.col("logradouro_work").rlike(r"^BECO\b"), F.lit("BECO"))
         .when(F.col("logradouro_work").rlike(r"^ESCADARIA\b"), F.lit("ESCADARIA"))
         .when(F.col("logradouro_work").rlike(r"^TRAVESSA\b|^TV\b"), F.lit("TRAVESSA"))
         .when(F.col("logradouro_work").rlike(r"^RODOVIA\b|^ROD\b"), F.lit("RODOVIA"))
         .when(F.col("logradouro_work").rlike(r"^ESTRADA\b"), F.lit("ESTRADA"))
         .when(F.col("logradouro_work").rlike(r"^ALAMEDA\b"), F.lit("ALAMEDA"))
         .when(F.col("logradouro_work").rlike(r"^PRACA\b"), F.lit("PRACA"))
         .when(F.col("logradouro_work").rlike(r"^LADEIRA\b"), F.lit("LADEIRA"))
         .when(F.col("logradouro_work").rlike(r"^VIELA\b"), F.lit("VIELA"))
         .when(F.col("logradouro_work").rlike(r"^CORREGO\b"), F.lit("CORREGO"))
         .when(F.col("logradouro_work").rlike(r"^SITIO\b"), F.lit("SITIO"))
         .when(F.col("logradouro_work").rlike(r"^FAZENDA\b"), F.lit("FAZENDA"))
         .when(F.col("logradouro_work").rlike(r"^BR\s+[0-9]+"), F.lit("RODOVIA"))
    )

    .withColumn(
        "logradouro_nome_norm",
        F.regexp_replace(
            F.col("logradouro_work"),
            r"^(RUA|R|AVENIDA|AV|BECO|ESCADARIA|TRAVESSA|TV|RODOVIA|ROD|ESTRADA|ALAMEDA|PRACA|LADEIRA|VIELA|CORREGO|SITIO|FAZENDA)\b\s*",
            ""
        )
    )
    .withColumn("logradouro_nome_norm", F.regexp_replace(F.col("logradouro_nome_norm"), r"[^A-Z0-9 ]", " "))
    .withColumn("logradouro_nome_norm", F.regexp_replace(F.col("logradouro_nome_norm"), r"\s+", " "))
    .withColumn("logradouro_nome_norm", F.trim(F.col("logradouro_nome_norm")))
    .withColumn(
        "logradouro_nome_norm",
        F.when(F.col("logradouro_nome_norm") == "", F.lit(None)).otherwise(F.col("logradouro_nome_norm"))
    )
)

# -----------------------------------------------------------------------------
# Parse técnico de faixa/paridade do complemento do CEP Aberto
# -----------------------------------------------------------------------------

df_cepaberto_es = (
    df_cepaberto_es
    .withColumn(
        "cep_lado",
        F.when(F.col("complemento_norm").rlike(r"LADO\s+IMPAR"), F.lit("IMPAR"))
         .when(F.col("complemento_norm").rlike(r"LADO\s+PAR"), F.lit("PAR"))
    )
    .withColumn(
        "cep_num_ini_de",
        F.regexp_extract(F.col("complemento_norm"), r"\bDE\s+([0-9]{1,6})\b", 1)
    )
    .withColumn(
        "cep_num_fim_ate",
        F.regexp_extract(F.col("complemento_norm"), r"\bATE\s+([0-9]{1,6})\b", 1)
    )
    .withColumn(
        "cep_num_fim_a",
        F.regexp_extract(F.col("complemento_norm"), r"\bA\s+([0-9]{1,6})\b", 1)
    )
    .withColumn(
        "cep_num_ini",
        F.when(F.col("cep_num_ini_de") != "", F.col("cep_num_ini_de").cast("int"))
    )
    .withColumn(
        "cep_num_fim",
        F.when(F.col("cep_num_fim_ate") != "", F.col("cep_num_fim_ate").cast("int"))
         .when(F.col("cep_num_fim_a") != "", F.col("cep_num_fim_a").cast("int"))
    )
    .drop("cep_num_ini_de", "cep_num_fim_ate", "cep_num_fim_a")
    .withColumn(
        "fl_cep_tem_faixa_numero",
        F.when(
            F.col("cep_num_ini").isNotNull()
            | F.col("cep_num_fim").isNotNull()
            | F.col("cep_lado").isNotNull(),
            F.lit("S")
        ).otherwise(F.lit("N"))
    )
)

# -----------------------------------------------------------------------------
# ID técnico da referência
# -----------------------------------------------------------------------------

df_cepaberto_es = (
    df_cepaberto_es
    .withColumn(
        "id_cepaberto_es",
        F.substring(
            F.md5(
                F.concat_ws(
                    "|",
                    F.coalesce(F.col("cep"), F.lit("[NULL]")),
                    F.coalesce(F.col("logradouro_raw"), F.lit("[NULL]")),
                    F.coalesce(F.col("complemento_raw"), F.lit("[NULL]")),
                    F.coalesce(F.col("bairro_raw"), F.lit("[NULL]")),
                    F.coalesce(F.col("cidade_cepaberto_id"), F.lit("[NULL]")),
                    F.coalesce(F.col("estado_cepaberto_id"), F.lit("[NULL]"))
                )
            ),
            1,
            30
        )
    )
    .withColumn("dt_carga_bronze", F.current_timestamp())
    .select(
        "id_cepaberto_es",
        "cep",
        "logradouro_raw",
        "complemento_raw",
        "bairro_raw",
        "cidade_cepaberto_id",
        "estado_cepaberto_id",
        "logradouro_norm",
        "complemento_norm",
        "bairro_norm",
        "tp_logradouro_norm",
        "logradouro_nome_norm",
        "cep_lado",
        "cep_num_ini",
        "cep_num_fim",
        "fl_cep_tem_faixa_numero",
        "arquivo_csv_origem",
        "dt_carga_bronze"
    )
    .dropDuplicates(["id_cepaberto_es"])
)

df_cepaberto_es = df_cepaberto_es.persist(StorageLevel.MEMORY_AND_DISK)

# -----------------------------------------------------------------------------
# Limpeza segura do catálogo e do path HDFS
# -----------------------------------------------------------------------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_destino}")
spark.sql(f"DROP TABLE IF EXISTS {tabela_full}")

hadoop_conf = spark._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(path_destino)

if fs.exists(hdfs_path):
    fs.delete(hdfs_path, True)

# -----------------------------------------------------------------------------
# Gravação + registro no Hive Metastore pelo Spark
# -----------------------------------------------------------------------------

(
    df_cepaberto_es
    .write
    .mode("overwrite")
    .format("parquet")
    .option("path", path_destino)
    .saveAsTable(tabela_full)
)

spark.sql(f"REFRESH TABLE {tabela_full}")

# -----------------------------------------------------------------------------
# Invalidação opcional no Impala, se houver cursor aberto no notebook
# -----------------------------------------------------------------------------

if "cursor" in globals():
    try:
        cursor.execute(f"INVALIDATE METADATA {tabela_full}")
        cursor.execute(f"COMPUTE STATS {tabela_full}")
        print(f"[OK] Metadados Impala invalidados e estatísticas atualizadas: {tabela_full}")
    except Exception as e:
        print(f"[AVISO] Tabela criada no Hive/Spark, mas Impala não foi atualizado via cursor: {str(e)}")

# -----------------------------------------------------------------------------
# Validação mínima
# -----------------------------------------------------------------------------

df_check = spark.table(tabela_full)

print("=" * 120)
print("BRONZE CEP ABERTO ES CRIADA")
print("=" * 120)
print(f"Tabela : {tabela_full}")
print(f"Path   : {path_destino}")

df_check.agg(
    F.count("*").alias("qtd_registros"),
    F.countDistinct("id_cepaberto_es").alias("qtd_ids_distintos"),
    F.countDistinct("cep").alias("qtd_ceps_distintos"),
    F.countDistinct("cidade_cepaberto_id").alias("qtd_cidades_cepaberto"),
    F.countDistinct("bairro_norm").alias("qtd_bairros"),
    F.countDistinct("logradouro_nome_norm").alias("qtd_logradouros"),
    F.sum(F.when(F.col("fl_cep_tem_faixa_numero") == "S", 1).otherwise(0)).alias("qtd_com_faixa_numero")
).show(truncate=False)

df_check.orderBy("cep").show(30, truncate=False)

df_cepaberto_es.unpersist()

CONFIGURAÇÃO DA CARGA
Diretório CSV : /home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR
Tabela destino: bronze.tbl_cep_es
Path destino  : /data_lake/bronze/intlpris/tbl_cep_es

ARQUIVOS CSV LOCALIZADOS
/home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR/es.cepaberto_parte_1.csv
/home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR/es.cepaberto_parte_2.csv
/home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR/es.cepaberto_parte_3.csv
/home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR/es.cepaberto_parte_4.csv
/home/diogo.silva/sejus-sinp/00_ETL/AUXILIAR/es.cepaberto_parte_5.csv


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/cloudera/parcels/SPARK3/lib/spark3/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/cloudera/parcels/SPARK3/lib/spark3/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib64/python3.6/socket.py", line 586, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.storagelevel import StorageLevel

import time
import requests

# =============================================================================
# GEOLOCALIZAÇÃO INCREMENTAL SIMPLES - BRONZE.TBL_CEP_ES
#
# Regra:
# - Se bronze.tbl_cep_es não tiver lat/long, cria como null.
# - Seleciona até 8000 CEPs distintos ainda sem lat/long.
# - Consulta CEP Aberto.
# - Regrava bronze.tbl_cep_es com os valores.
#
# Não usa UPDATE.
# Não usa write_impala_table_partioned.
# =============================================================================

# -----------------------------------------------------------------------------
# Configurações
# -----------------------------------------------------------------------------

schema_destino = "bronze"
tabela_destino = "tbl_cep_es"
tabela_full = f"{schema_destino}.{tabela_destino}"

path_bronze = "/data_lake/bronze/intlpris/"
path_destino = f"{path_bronze.rstrip('/')}/{tabela_destino}"
path_tmp = f"{path_bronze.rstrip('/')}/tmp_{tabela_destino}_geo"

CEP_ABERTO_TOKEN = "477c2f862a8c8c25f99a5f7ea002f000"

url_cepaberto_cep = "https://www.cepaberto.com/api/v3/cep"

headers_cepaberto = {
    "Authorization": f"Token token={CEP_ABERTO_TOKEN}"
}

# API permite 1 requisição por segundo. Usar 1.10 para margem.
tempo_espera_segundos = 1.10

timeout_segundos = 20
max_tentativas = 3

# Lote diário seguro
limite_ceps_execucao = 8000

# Se True, reconsulta CEPs já consultados que continuam sem lat/long.
# Se False, consulta apenas CEPs sem dt_consulta_geo/status anterior.
reconsultar_ja_consultados_sem_geo = False

# -----------------------------------------------------------------------------
# HDFS / catálogo
# -----------------------------------------------------------------------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_destino}")
spark.sql(f"REFRESH TABLE {tabela_full}")

hadoop_conf = spark._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)

# -----------------------------------------------------------------------------
# Carregar tabela
# -----------------------------------------------------------------------------

df_base_original = spark.table(tabela_full)

# -----------------------------------------------------------------------------
# Garantir campos de geolocalização
# -----------------------------------------------------------------------------

df_base = df_base_original

if "lat" not in df_base.columns:
    if "latitude" in df_base.columns:
        df_base = df_base.withColumn("lat", F.col("latitude").cast("double"))
    else:
        df_base = df_base.withColumn("lat", F.lit(None).cast("double"))

if "long" not in df_base.columns:
    if "longitude" in df_base.columns:
        df_base = df_base.withColumn("long", F.col("longitude").cast("double"))
    else:
        df_base = df_base.withColumn("long", F.lit(None).cast("double"))

if "fl_geo_consultado" not in df_base.columns:
    df_base = df_base.withColumn("fl_geo_consultado", F.lit(None).cast("string"))

if "fl_geo_encontrada" not in df_base.columns:
    df_base = df_base.withColumn("fl_geo_encontrada", F.lit(None).cast("string"))

if "precisao_geo" not in df_base.columns:
    df_base = df_base.withColumn("precisao_geo", F.lit(None).cast("string"))

if "geo_bairro" not in df_base.columns:
    df_base = df_base.withColumn("geo_bairro", F.lit(None).cast("string"))

if "geo_logradouro" not in df_base.columns:
    df_base = df_base.withColumn("geo_logradouro", F.lit(None).cast("string"))

if "geo_cidade" not in df_base.columns:
    df_base = df_base.withColumn("geo_cidade", F.lit(None).cast("string"))

if "geo_uf" not in df_base.columns:
    df_base = df_base.withColumn("geo_uf", F.lit(None).cast("string"))

if "status_http_geo" not in df_base.columns:
    df_base = df_base.withColumn("status_http_geo", F.lit(None).cast("int"))

if "erro_consulta_geo" not in df_base.columns:
    df_base = df_base.withColumn("erro_consulta_geo", F.lit(None).cast("string"))

if "dt_consulta_geo" not in df_base.columns:
    df_base = df_base.withColumn("dt_consulta_geo", F.lit(None).cast("string"))

df_base = (
    df_base
    .withColumn("cep", F.regexp_replace(F.col("cep").cast("string"), r"[^0-9]", ""))
    .filter(F.length("cep") == 8)
)

df_base = df_base.persist(StorageLevel.MEMORY_AND_DISK)

# -----------------------------------------------------------------------------
# Selecionar CEPs pendentes
# -----------------------------------------------------------------------------

cond_sem_geo = (
    F.col("lat").isNull()
    | F.col("long").isNull()
)

if reconsultar_ja_consultados_sem_geo:
    cond_pendente = cond_sem_geo
else:
    cond_pendente = (
        cond_sem_geo
        & (
            F.col("dt_consulta_geo").isNull()
            | (F.col("fl_geo_consultado") != "S")
        )
    )

df_ceps_pendentes = (
    df_base
    .filter(cond_pendente)
    .select("cep")
    .dropDuplicates(["cep"])
    .orderBy("cep")
    .limit(limite_ceps_execucao)
)

ceps_pendentes = [r["cep"] for r in df_ceps_pendentes.collect()]

print("=" * 120)
print("GEOLOCALIZAÇÃO INCREMENTAL - bronze.tbl_cep_es")
print("=" * 120)
print(f"Tabela               : {tabela_full}")
print(f"Path                 : {path_destino}")
print(f"CEPs no lote          : {len(ceps_pendentes)}")
print(f"Limite execução       : {limite_ceps_execucao}")
print(f"Intervalo requisição  : {tempo_espera_segundos}s")
print(f"Reconsulta sem geo    : {'S' if reconsultar_ja_consultados_sem_geo else 'N'}")

# -----------------------------------------------------------------------------
# Função de consulta
# -----------------------------------------------------------------------------

def parse_float(valor):
    if valor is None:
        return None
    try:
        return float(str(valor).replace(",", "."))
    except:
        return None


def consultar_cep(cep):
    ultimo_status = None
    ultimo_erro = None

    for tentativa in range(1, max_tentativas + 1):
        try:
            response = requests.get(
                url_cepaberto_cep,
                headers=headers_cepaberto,
                params={"cep": cep},
                timeout=timeout_segundos
            )

            ultimo_status = response.status_code

            if response.status_code == 200:
                try:
                    data = response.json()
                except Exception as e:
                    return {
                        "cep": cep,
                        "lat_novo": None,
                        "long_novo": None,
                        "fl_geo_consultado_novo": "S",
                        "fl_geo_encontrada_novo": "N",
                        "precisao_geo_novo": "JSON_INVALIDO",
                        "geo_bairro_novo": None,
                        "geo_logradouro_novo": None,
                        "geo_cidade_novo": None,
                        "geo_uf_novo": None,
                        "status_http_geo_novo": response.status_code,
                        "erro_consulta_geo_novo": f"JSON_INVALIDO: {str(e)}",
                        "dt_consulta_geo_novo": time.strftime("%Y-%m-%d %H:%M:%S")
                    }

                if not data:
                    return {
                        "cep": cep,
                        "lat_novo": None,
                        "long_novo": None,
                        "fl_geo_consultado_novo": "S",
                        "fl_geo_encontrada_novo": "N",
                        "precisao_geo_novo": "NAO_ENCONTRADO",
                        "geo_bairro_novo": None,
                        "geo_logradouro_novo": None,
                        "geo_cidade_novo": None,
                        "geo_uf_novo": None,
                        "status_http_geo_novo": response.status_code,
                        "erro_consulta_geo_novo": None,
                        "dt_consulta_geo_novo": time.strftime("%Y-%m-%d %H:%M:%S")
                    }

                cidade = data.get("cidade") or {}
                estado = data.get("estado") or {}

                lat = parse_float(data.get("latitude"))
                lon = parse_float(data.get("longitude"))

                geo_logradouro = data.get("logradouro")
                geo_bairro = data.get("bairro")

                if lat is not None and lon is not None:
                    if geo_logradouro:
                        precisao_geo = "CEP_LOGRADOURO"
                    elif geo_bairro:
                        precisao_geo = "CEP_BAIRRO"
                    else:
                        precisao_geo = "CEP_CIDADE"
                else:
                    precisao_geo = "SEM_COORDENADA"

                return {
                    "cep": cep,
                    "lat_novo": lat,
                    "long_novo": lon,
                    "fl_geo_consultado_novo": "S",
                    "fl_geo_encontrada_novo": "S" if lat is not None and lon is not None else "N",
                    "precisao_geo_novo": precisao_geo,
                    "geo_bairro_novo": geo_bairro,
                    "geo_logradouro_novo": geo_logradouro,
                    "geo_cidade_novo": cidade.get("nome"),
                    "geo_uf_novo": estado.get("sigla"),
                    "status_http_geo_novo": response.status_code,
                    "erro_consulta_geo_novo": None,
                    "dt_consulta_geo_novo": time.strftime("%Y-%m-%d %H:%M:%S")
                }

            if response.status_code == 403:
                ultimo_erro = f"HTTP_403_FORBIDDEN: {response.text[:300]}"
                time.sleep(tempo_espera_segundos * tentativa * 3)
            else:
                ultimo_erro = f"HTTP_{response.status_code}: {response.text[:300]}"
                time.sleep(tempo_espera_segundos * tentativa)

        except Exception as e:
            ultimo_erro = str(e)
            time.sleep(tempo_espera_segundos * tentativa)

    return {
        "cep": cep,
        "lat_novo": None,
        "long_novo": None,
        "fl_geo_consultado_novo": "N",
        "fl_geo_encontrada_novo": "N",
        "precisao_geo_novo": "ERRO_CONSULTA",
        "geo_bairro_novo": None,
        "geo_logradouro_novo": None,
        "geo_cidade_novo": None,
        "geo_uf_novo": None,
        "status_http_geo_novo": ultimo_status,
        "erro_consulta_geo_novo": ultimo_erro,
        "dt_consulta_geo_novo": time.strftime("%Y-%m-%d %H:%M:%S")
    }

# -----------------------------------------------------------------------------
# Executar lote
# -----------------------------------------------------------------------------

resultados = []

inicio = time.time()

for i, cep in enumerate(ceps_pendentes, start=1):
    resultados.append(consultar_cep(cep))

    if i % 50 == 0 or i == len(ceps_pendentes):
        print(f"[INFO] CEPs consultados: {i}/{len(ceps_pendentes)} | tempo: {round(time.time() - inicio, 2)}s")

    if i < len(ceps_pendentes):
        time.sleep(tempo_espera_segundos)

# -----------------------------------------------------------------------------
# DataFrame do lote
# -----------------------------------------------------------------------------

schema_resultado = StructType([
    StructField("cep", StringType(), True),
    StructField("lat_novo", DoubleType(), True),
    StructField("long_novo", DoubleType(), True),
    StructField("fl_geo_consultado_novo", StringType(), True),
    StructField("fl_geo_encontrada_novo", StringType(), True),
    StructField("precisao_geo_novo", StringType(), True),
    StructField("geo_bairro_novo", StringType(), True),
    StructField("geo_logradouro_novo", StringType(), True),
    StructField("geo_cidade_novo", StringType(), True),
    StructField("geo_uf_novo", StringType(), True),
    StructField("status_http_geo_novo", IntegerType(), True),
    StructField("erro_consulta_geo_novo", StringType(), True),
    StructField("dt_consulta_geo_novo", StringType(), True)
])

if resultados:
    df_resultado = spark.createDataFrame(resultados, schema=schema_resultado)
else:
    df_resultado = spark.createDataFrame([], schema=schema_resultado)

df_resultado = (
    df_resultado
    .withColumn("cep", F.regexp_replace(F.col("cep").cast("string"), r"[^0-9]", ""))
    .filter(F.length("cep") == 8)
    .dropDuplicates(["cep"])
)

df_resultado = df_resultado.persist(StorageLevel.MEMORY_AND_DISK)

# -----------------------------------------------------------------------------
# Atualizar valores em DataFrame final
# -----------------------------------------------------------------------------

campos_atualizados = [
    "lat",
    "long",
    "fl_geo_consultado",
    "fl_geo_encontrada",
    "precisao_geo",
    "geo_bairro",
    "geo_logradouro",
    "geo_cidade",
    "geo_uf",
    "status_http_geo",
    "erro_consulta_geo",
    "dt_consulta_geo"
]

campos_preservar = [c for c in df_base.columns if c not in campos_atualizados]

df_final = (
    df_base.alias("b")
    .join(df_resultado.alias("g"), "cep", "left")
    .select(
        *[F.col(f"b.{c}").alias(c) for c in campos_preservar],

        F.coalesce(F.col("g.lat_novo"), F.col("b.lat")).alias("lat"),
        F.coalesce(F.col("g.long_novo"), F.col("b.long")).alias("long"),

        F.when(
            F.col("g.cep").isNotNull(),
            F.col("g.fl_geo_consultado_novo")
        ).otherwise(F.col("b.fl_geo_consultado")).alias("fl_geo_consultado"),

        F.when(
            F.col("g.cep").isNotNull(),
            F.col("g.fl_geo_encontrada_novo")
        ).otherwise(F.col("b.fl_geo_encontrada")).alias("fl_geo_encontrada"),

        F.coalesce(F.col("g.precisao_geo_novo"), F.col("b.precisao_geo")).alias("precisao_geo"),
        F.coalesce(F.col("g.geo_bairro_novo"), F.col("b.geo_bairro")).alias("geo_bairro"),
        F.coalesce(F.col("g.geo_logradouro_novo"), F.col("b.geo_logradouro")).alias("geo_logradouro"),
        F.coalesce(F.col("g.geo_cidade_novo"), F.col("b.geo_cidade")).alias("geo_cidade"),
        F.coalesce(F.col("g.geo_uf_novo"), F.col("b.geo_uf")).alias("geo_uf"),

        F.coalesce(F.col("g.status_http_geo_novo"), F.col("b.status_http_geo")).alias("status_http_geo"),
        F.coalesce(F.col("g.erro_consulta_geo_novo"), F.col("b.erro_consulta_geo")).alias("erro_consulta_geo"),
        F.coalesce(F.col("g.dt_consulta_geo_novo"), F.col("b.dt_consulta_geo")).alias("dt_consulta_geo")
    )
)

df_final = df_final.persist(StorageLevel.MEMORY_AND_DISK)

qtd_final = df_final.count()

# -----------------------------------------------------------------------------
# Materializar em path temporário antes de derrubar/recriar a tabela
# -----------------------------------------------------------------------------

tmp_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(path_tmp)

if fs.exists(tmp_hdfs_path):
    fs.delete(tmp_hdfs_path, True)

(
    df_final
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .parquet(path_tmp)
)

df_final_materializado = spark.read.parquet(path_tmp)

# -----------------------------------------------------------------------------
# Regravar bronze.tbl_cep_es
# -----------------------------------------------------------------------------

spark.sql(f"DROP TABLE IF EXISTS {tabela_full}")

dest_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(path_destino)

if fs.exists(dest_hdfs_path):
    fs.delete(dest_hdfs_path, True)

(
    df_final_materializado
    .write
    .mode("overwrite")
    .format("parquet")
    .option("path", path_destino)
    .option("compression", "snappy")
    .saveAsTable(tabela_full)
)

spark.sql(f"REFRESH TABLE {tabela_full}")

# -----------------------------------------------------------------------------
# Atualizar Impala, se houver cursor
# -----------------------------------------------------------------------------

if "cursor" in globals():
    try:
        cursor.execute(f"INVALIDATE METADATA {tabela_full}")
        cursor.execute(f"COMPUTE STATS {tabela_full}")
        print(f"[OK] Metadados Impala atualizados: {tabela_full}")
    except Exception as e:
        print(f"[AVISO] Spark/Hive atualizado, mas Impala não foi atualizado via cursor: {str(e)}")

# -----------------------------------------------------------------------------
# Resumo final
# -----------------------------------------------------------------------------

df_check = spark.table(tabela_full)

print("=" * 120)
print("BRONZE.TBL_CEP_ES ATUALIZADA COM LAT/LONG")
print("=" * 120)
print(f"Tabela         : {tabela_full}")
print(f"Path           : {path_destino}")
print(f"Registros final: {qtd_final}")

df_check.agg(
    F.count("*").alias("qtd_registros"),
    F.countDistinct("cep").alias("qtd_ceps_distintos"),
    F.countDistinct(F.when(F.col("lat").isNotNull() & F.col("long").isNotNull(), F.col("cep"))).alias("qtd_ceps_com_lat_long"),
    F.countDistinct(F.when(F.col("lat").isNull() | F.col("long").isNull(), F.col("cep"))).alias("qtd_ceps_sem_lat_long"),
    F.sum(F.when(F.col("fl_geo_consultado") == "S", 1).otherwise(0)).alias("qtd_registros_geo_consultado"),
    F.sum(F.when(F.col("fl_geo_encontrada") == "S", 1).otherwise(0)).alias("qtd_registros_geo_encontrada")
).show(truncate=False)

# -----------------------------------------------------------------------------
# Limpeza
# -----------------------------------------------------------------------------

df_base.unpersist()
df_resultado.unpersist()
df_final.unpersist()

if fs.exists(tmp_hdfs_path):
    fs.delete(tmp_hdfs_path, True)

print("[OK] Processo concluído.")